# Project 2: Heatwave analysis for Italy

Analysis of the summer 2022 heatwave in Italy: ERA5 2-metre temperature (Copernicus CDS API), weekly all-cause mortality (World Mortality Dataset by Karlinsky & Kobak), a simple time-series model of excess mortality, and a binned scatter plot of the mortality–temperature relationship.

**Requirements:** see `requirements.txt` and the project `README.md`. CDS API credentials must be set in `~/.cdsapirc`, and `world_mortality.csv` must be placed in `data/`. Expected layout: this notebook lives in `notebooks/`; the ERA5 NetCDF files are downloaded to `data/era5/` (not tracked by git).

### Task 1 : Generate the ERA5 data extract for Italy

Task 1.1: Generate the ERA5 data extract for Italy using the API of daily-hourly data starting in 2015. As a consequence, the reference period for our analysis will be 2015-2020, when we assess the situation in the summer of 2022. Recall that the coordinates are (anti-clockwise): [N,W,S,E]. The bounding box for Italy is [46,9,37,18.5,]

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import os

In [ ]:
# Retrieve ERA5 only for years not already present locally.
import cdsapi
import warnings
warnings.filterwarnings("ignore")
DATADIR = "../data/era5"
os.makedirs(DATADIR, exist_ok=True)
c = cdsapi.Client()
request = {
    "product_type":"reanalysis", "variable":"2m_temperature",
    "month":[f"{m:02d}" for m in range(1,13)],
    "day":[f"{d:02d}" for d in range(1,32)],
    "time":[f"{h:02d}:00" for h in range(24)], "area":[46,9,37,18.5],
    "data_format":"netcdf", "download_format":"unarchived"
}
for year in range(2015,2025):
    target=os.path.join(DATADIR,f"Italy_{year}.nc")
    if os.path.exists(target):
        print(f"Already downloaded: {year}"); continue
    print(f"Downloading {year}...")
    c.retrieve("reanalysis-era5-single-levels",{**request,"year":str(year)},target)
print("Download check complete")


In [ ]:
# Loading and preparing ERA5 temperature data

# The data were downloaded year by year to comply with CDS API limits,
# then merged using open_mfdataset() to reconstruct the complete series.

import xarray as xr

#  load the files
ds = xr.open_mfdataset(f"{DATADIR}/Italy_*.nc")

# Conversion to Celsius directly
t2m_C = ds["t2m"] - 273.15

# Load into memory
t2m_C = t2m_C.load()

# Displaying the converted data
print(t2m_C)

The 2-metre temperature data (ERA5) were loaded for 2015–2024 using `xarray.open_mfdataset`.
The dataset is structured along the time and spatial dimensions (latitude, longitude) and stored as a dask array to optimise memory usage.

Task 1.2: Then aggregate hourly local data for the week, and then aggregate the local measures.
          Display the map. Finally, write the data to disk as a csv file.

In [ ]:
# Daily aggregation
# We group by date (Year-Month-Day) to calculate the daily average temperature
year_month_day_mean = t2m_C.groupby("valid_time.date").mean()

# Display the aggregated data
print(year_month_day_mean)

In [ ]:
# Latitude weights (cosine of latitude). This averages the full bounding box,
# including sea and neighboring regions; no Italy land mask is applied.

weights = np.cos(np.deg2rad(year_month_day_mean["latitude"]))
weights = weights.broadcast_like(year_month_day_mean)
weights.name = "weights"

In [ ]:
# weighted mean

year_month_day_mean_weighted = year_month_day_mean.weighted(weights)

Italy_t2m = year_month_day_mean_weighted.mean(
    dim=["latitude", "longitude"]
)

In [ ]:
print(year_month_day_mean)

In [ ]:
import datetime

# Extraction for August 3rd, 2022
# We create a datetime.date object to match the index format
target_date = datetime.date(2022, 8, 3)

map_0308 = year_month_day_mean.sel(date=target_date)

# Check the result
print(map_0308)

In [ ]:
os.makedirs("pics", exist_ok=True)

fig, ax = plt.subplots(
    1, 1,
    figsize=(16, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

im = ax.pcolormesh(
    map_0308.longitude,
    map_0308.latitude,
    map_0308,
    cmap='RdBu_r',
    vmin=map_0308.min(),
    vmax=map_0308.max()
)

ax.set_title('Mean temperatures on 03.08.2022', fontsize=16)

ax.gridlines(draw_labels=True, linewidth=1,
             color='gray', alpha=0.5, linestyle='--')

ax.coastlines(color='black')

ax.set_extent([9, 18.5, 37, 46], crs=ccrs.PlateCarree())

cbar = plt.colorbar(im, fraction=0.05, pad=0.04)
cbar.set_label('temperature (°C)')

fig.tight_layout()

fig.savefig("pics/map_Italy_August2022_2.png", bbox_inches='tight')

plt.show()

### Aggregation of weekly mean temperatures

In [ ]:
# Aggregate using ISO year and ISO week so weeks around New Year stay aligned.
df_italy = Italy_t2m.to_dataframe(name="t2m").reset_index()
df_italy["date"] = pd.to_datetime(df_italy["date"])
iso = df_italy["date"].dt.isocalendar()
df_italy["year"] = iso.year.astype(int)
df_italy["week"] = iso.week.astype(int)
df = df_italy.groupby(["year","week"],as_index=False)["t2m"].mean()
df["start_date"] = pd.to_datetime(df["year"].astype(str)+"-W"+df["week"].astype(str).str.zfill(2)+"-1",format="%G-W%V-%u")
df.head()


# Task 1.3

load your csv-file of the temperature data, and visualise this temperature data. Consider the summer of 2022 starting in June and finishing end of August.

In [ ]:
# Filtering data for Summer 2022
summer_2022 = df[
    (df["year"] == 2022) &
    (df["week"].between(22, 35))
]
summer_2022

In [ ]:
# Plot weekly temperatures using the ISO-week Monday already computed above.
plt.figure(figsize=(14, 6))
plt.plot(df["start_date"], df["t2m"], color="black", linewidth=1)
plt.axvspan(pd.to_datetime("2022-06-01"), pd.to_datetime("2022-08-31"), color="darkgrey", alpha=0.6)
plt.grid(True, linestyle="-", alpha=0.3)
plt.ylabel("Temperature (°C)", fontsize=14)
plt.xlabel("Week")
plt.title("Weekly mean temperature in Italy, 2015–2024")
plt.xlim(df["start_date"].min(), df["start_date"].max())
plt.show()


# 2 The mortality data

Task 2.1. Use of the World Mortality Dataset compiled by Ariel Karlinsky and Dmitry Kobak that records all-cause mortality at a weekly level. Filter Italy, which has iso3c code "ITA".

In [ ]:
# Loading and previewing the mortality dataset
file_path = "../data/world_mortality.csv"
mortality= pd.read_csv(file_path)
mortality.head()

In [ ]:
# Filtering Italy data and cleaning the mortality dataset
country = "Italy"  # iso3c == "ITA"

# Transform the raw world mortality data into `mortality`
mortality = (
    mortality[mortality["iso3c"] == "ITA"]      # filter
    .drop(columns=["iso3c", "country_name", "time_unit"]) # select(-c(...))
    .rename(columns={"deaths": "mortality", "time": "week"}) # rename
)

In [ ]:
# Create the start-of-week date (equivalent to get_date)
# %G-%V-%u is used to follow the ISO week-date standard
mortality["start_date"] = pd.to_datetime(
    mortality["year"].astype(str) + "-" +
    mortality["week"].astype(str) + "-1",
    format="%G-%V-%u"
)

print(mortality)

# 3 A simple time-series model of excess mortality

Task 3: Next, estimate a simple time-series model of excess mortality for the period before 2020-03-01, which only considers a linear year effect and week fixed effects. Then use the model to predict mortality for the summer of 2022, and compute the measure of excess mortality (i.e. observed - predicted weekly mortality from all causes).

In [ ]:
from sklearn.linear_model import LinearRegression

# 1. Create dummy variables for the weeks
# Keep 'year' as is for the linear effect
df_model = pd.get_dummies(mortality, columns=['week'], drop_first=True)

# 2. Split train (before March 2020) / test (summer 2022)
train = df_model[df_model["start_date"] < "2020-03-01"].copy()
summer_2022 = df_model[(df_model["year"] == 2022) & (mortality["week"].between(22, 35))].copy()

# Define the explanatory variables (X) and the target (y)
# X contains 'year' and all the 'week_2', 'week_3', ... columns
features = [col for col in df_model.columns if col.startswith('week_') or col == 'year']
X_train = train[features]
y_train = train["mortality"]

In [ ]:
# 3. Fit the model
model = LinearRegression()
model.fit(X_train, y_train)

# 4. Predict summer 2022
X_summer = summer_2022[features]
summer_2022["predicted"] = model.predict(X_summer)

# 5. Compute excess mortality (Observed - Predicted)
summer_2022["excess_mortality"] = summer_2022["mortality"] - summer_2022["predicted"]

# Total excess mortality
total_excess = summer_2022["excess_mortality"].sum()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

#1. Figure creation
plt.figure(figsize=(10, 10))

# 2. Plotting the excess mortality curve
plt.plot(summer_2022["start_date"], summer_2022["excess_mortality"],
         color="#FF2E00", linewidth=2, label="excess deaths")

# 3.Axis configuration
plt.title("Italy : Excess deaths", loc='left', fontsize=22, pad=20)
plt.ylabel("death count", fontsize=16)
plt.xlabel("date", fontsize=16)

# 4. Format the X-axis to display Jul, Aug, Sep
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

# 5. Grid style
plt.grid(True, which='both', color='lightgrey', linestyle='-', linewidth=1)
ax.set_facecolor('white')
for spine in ax.spines.values():
    spine.set_edgecolor('black')

# 6. Legend
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1),
           frameon=False, fontsize=14, ncol=1)

plt.tight_layout()
plt.show()

In [ ]:
print(f"Estimated total excess mortality (summer 2022): {total_excess:.0f} deaths")

# 4 The mortality and temperature relationship

Task 4: Finally consider the (potential all-cause) mortality and temperature relationship using a bin scatter plot

In [ ]:
# Join on the actual Monday date, not on row position or assumed row order.
df_merged = mortality[["start_date","year","week","mortality"]].merge(
    df[["start_date","t2m"]],on="start_date",how="inner",validate="one_to_one")
df_merged = df_merged[df_merged["year"]!=2024].dropna(subset=["t2m","mortality"])
df_merged.head()


In [ ]:
from binsreg import binsreg
import matplotlib.pyplot as plt

# 2. Generate the binned scatterplot
# y: Dependent variable "mortality"
# x: Independent variable "t2m"
# nbins: Number of bins
# line=(1,0):
# The first '1' is the degree of the polynomial (1 = linear/straight line)
# The second '0' means 0 constraints on derivatives at the knots (creates the gaps)
est = binsreg(y=df_merged['mortality'], x=df_merged['t2m'], nbins=10, line=(1,0))